In [1]:
import requests
import pandas as pd
import joblib
import numpy as np
import re

BASE_URL = "https://sdp-prem-prod.premier-league-prod.pulselive.com/api/v2/matches"
competition_id = 8
season_id = 2025

output_all = "../data/raw/football_data/premier_league_2025_26_fixtures.csv"
output_upcoming = "../data/processed/premier_league_2025_26_upcoming_prediction_template.csv"
output_annotated = "../data/processed/premier_league_2025_26_upcoming_with_teams_and_date.csv"
feature_path = "../models/weekly/homewin/feature_list_20260327T205139Z.pkl"
engineered_history_path = "../data/processed/engineered_train_features.csv"

TEAM_MAP = {
    'manchester united': 'Man United',
    'man utd': 'Man United',
    'manchester city': 'Man City',
    'man city': 'Man City',
    'wolves': 'Wolves',
    'wolverhampton wanderers': 'Wolves',
    'nottingham forest': "Nott'm Forest",
    'west bromwich albion': 'West Brom',
    'west ham united': 'West Ham',
    'newcastle united': 'Newcastle',
    'tottenham hotspur': 'Tottenham',
    'leeds united': 'Leeds',
    'leicester city': 'Leicester',
    'sheffield united': 'Sheffield United',
    'crystal palace': 'Crystal Palace',
    'aston villa': 'Aston Villa',
    'brighton & hove albion': 'Brighton',
    'brighton and hove albion': 'Brighton',
    'bournemouth': 'Bournemouth',
    'liverpool': 'Liverpool',
    'chelsea': 'Chelsea',
    'arsenal': 'Arsenal',
    'brentford': 'Brentford',
    'burnley': 'Burnley',
    'luton town': 'Luton',
    'fulham': 'Fulham',
    'everton': 'Everton'
}
def standardize_team(val):
    if pd.isna(val):
        return val
    t = str(val).strip().lower()
    t = re.sub(r'\s+', ' ', t)
    return TEAM_MAP.get(t, val if isinstance(val, str) else str(val))

def clean_team_names(df, cols=["HomeTeam", "AwayTeam"]):
    for col in cols:
        if col in df.columns:
            df[col] = df[col].apply(standardize_team)
    return df

df_hist = pd.read_csv(engineered_history_path, low_memory=False)
df_hist["Date"] = pd.to_datetime(df_hist["Date"], errors="coerce")
if "Kickoff" not in df_hist.columns:
    df_hist["Kickoff"] = df_hist["Date"]
df_hist = clean_team_names(df_hist)

all_rows = []
for mw in range(1, 39):
    params = {
        "competition": competition_id,
        "season": season_id,
        "matchweek": mw,
        "_limit": 20
    }
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = requests.get(BASE_URL, params=params, headers=headers)
    js = resp.json()
    for match in js.get("data", []):
        home = match.get("homeTeam", {}).get("name", "")
        away = match.get("awayTeam", {}).get("name", "")
        kickoff = match.get("kickoff", "")
        matchweek = match.get("matchWeek", "")
        match_id = match.get("matchId", "")
        all_rows.append({
            "MatchWeek": matchweek,
            "HomeTeam": home,
            "AwayTeam": away,
            "FTHG": np.nan,   
            "FTAG": np.nan,
            "Kickoff": kickoff,
            "MatchId": match_id
        })

df_fixtures = pd.DataFrame(all_rows)
df_fixtures["Date"] = pd.to_datetime(df_fixtures["Kickoff"], errors="coerce")
df_fixtures = clean_team_names(df_fixtures)
df_fixtures.to_csv(output_all, index=False)

needed_cols = set(df_hist.columns)
missing_cols = needed_cols - set(df_fixtures.columns)
fixture_add = pd.DataFrame({col: np.nan for col in missing_cols}, index=df_fixtures.index)
df_fixtures = pd.concat([df_fixtures, fixture_add], axis=1)[df_hist.columns]
df_all = pd.concat([df_hist, df_fixtures], ignore_index=True)
df_all = df_all.sort_values(["Date", "HomeTeam", "AwayTeam"]).reset_index(drop=True)

def add_rolling_features(df, n_matches=5):
    df = df.copy()
    df = df.sort_values(["Date", "HomeTeam", "AwayTeam"]).reset_index(drop=True)
    df["FTHG"] = pd.to_numeric(df["FTHG"], errors="coerce")
    df["FTAG"] = pd.to_numeric(df["FTAG"], errors="coerce")
    df["HomeTeam_mean_FTHG"] = (
        df.groupby("HomeTeam")["FTHG"].apply(lambda x: x.shift(1).expanding().mean())
        .reset_index(level=0, drop=True)
    )
    df["AwayTeam_mean_FTAG"] = (
        df.groupby("AwayTeam")["FTAG"].apply(lambda x: x.shift(1).expanding().mean())
        .reset_index(level=0, drop=True)
    )
    records = []
    for idx, row in df.iterrows():
        past_home = df[
            (df["HomeTeam"] == row["HomeTeam"]) &
            (df["Date"] < row["Date"]) &
            df["FTHG"].notna()
        ].sort_values("Date").tail(n_matches)
        home_gf = past_home["FTHG"].mean() if not past_home.empty else np.nan
        home_ga = past_home["FTAG"].mean() if not past_home.empty else np.nan
        home_pts = past_home.apply(
            lambda r: 3 if r["FTHG"] > r["FTAG"] else (1 if r["FTHG"] == r["FTAG"] else 0), axis=1
        ).sum() if not past_home.empty else np.nan
        past_away = df[
            (df["AwayTeam"] == row["AwayTeam"]) &
            (df["Date"] < row["Date"]) &
            df["FTAG"].notna()
        ].sort_values("Date").tail(n_matches)
        away_gf = past_away["FTAG"].mean() if not past_away.empty else np.nan
        away_ga = past_away["FTHG"].mean() if not past_away.empty else np.nan
        away_pts = past_away.apply(
            lambda r: 3 if r["FTAG"] > r["FTHG"] else (1 if r["FTAG"] == r["FTHG"] else 0), axis=1
        ).sum() if not past_away.empty else np.nan
        records.append({
            "HomeRecentGF": home_gf,
            "HomeRecentGA": home_ga,
            "HomeRecentPts": home_pts,
            "AwayRecentGF": away_gf,
            "AwayRecentGA": away_ga,
            "AwayRecentPts": away_pts,
        })
    roll_df = pd.DataFrame(records, index=df.index)
    for col in roll_df.columns:
        df[col] = roll_df[col].values
    return df

df_all = add_rolling_features(df_all, n_matches=5)

mask_unplayed = (
    (df_all["FTHG"].isna() | (df_all["FTHG"].astype(str).str.strip() == "")) &
    (df_all["FTAG"].isna() | (df_all["FTAG"].astype(str).str.strip() == ""))
)
df_upcoming = df_all[mask_unplayed].copy()

feature_list = joblib.load(feature_path)
for col in feature_list:
    if col not in df_upcoming.columns:
        df_upcoming[col] = np.nan
df_model_ready = df_upcoming[feature_list].copy()
df_model_ready.to_csv(output_upcoming, index=False)

df_annotated = df_upcoming.copy()
df_annotated['Year'] = pd.to_datetime(df_annotated['Date'], errors="coerce").dt.year
df_annotated['Month'] = pd.to_datetime(df_annotated['Date'], errors="coerce").dt.month
df_annotated['DayOfWeek'] = pd.to_datetime(df_annotated['Date'], errors="coerce").dt.dayofweek
annotate_front = [c for c in ["HomeTeam", "AwayTeam", "Date", "Year", "Month", "DayOfWeek"] if c in df_annotated.columns]
df_annotated = df_annotated[annotate_front + [c for c in df_annotated.columns if c not in annotate_front]]

df_annotated.to_csv(output_annotated, index=False)

print(f"\nModel-ready prediction template saved to {output_upcoming}, shape: {df_model_ready.shape}")
print(f"Annotated template (with teams/dates/stats) saved to {output_annotated}, shape: {df_annotated.shape}")
print(df_annotated[["HomeTeam", "AwayTeam", "HomeTeam_mean_FTHG", "AwayTeam_mean_FTAG", "HomeRecentGF", "AwayRecentGF"]].head())

/var/folders/rj/ybzbgv510dd6rzmm3p9vf0000000gn/T/ipykernel_80976/1074282010.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_hist["Kickoff"] = df_hist["Date"]



Model-ready prediction template saved to ../data/processed/premier_league_2025_26_upcoming_prediction_template.csv, shape: (380, 183)
Annotated template (with teams/dates/stats) saved to ../data/processed/premier_league_2025_26_upcoming_with_teams_and_date.csv, shape: (380, 275)
         HomeTeam     AwayTeam  HomeTeam_mean_FTHG  AwayTeam_mean_FTAG  \
9339    Liverpool  Bournemouth            2.087607            1.223684   
9345  Aston Villa    Newcastle            1.366505            1.111369   
9346     Brighton       Fulham            1.311258            0.940476   
9347   Sunderland     West Ham            1.090226            1.088942   
9348    Tottenham      Burnley            1.829787            0.941520   

      HomeRecentGF  AwayRecentGF  
9339           2.8           1.4  
9345           1.8           1.0  
9346           2.0           1.2  
9347           1.0           1.6  
9348           1.6           1.2  
